# Домашнее задание 3. Парсинг, Git и тестирование на Python

**Цели задания:**

* Освоить базовые подходы к web-scraping с библиотеками `requests` и `BeautisulSoup`: навигация по страницам, извлечение HTML-элементов, парсинг.
* Научиться автоматизировать задачи с использованием библиотеки `schedule`.
* Попрактиковаться в использовании Git и оформлении проектов на GitHub.
* Написать и запустить простые юнит-тесты с использованием `pytest`.


В этом домашнем задании вы разработаете систему для автоматического сбора данных о книгах с сайта [Books to Scrape](http://books.toscrape.com). Нужно реализовать функции для парсинга всех страниц сайта, извлечения информации о книгах, автоматического ежедневного запуска задачи и сохранения результата.

Важной частью задания станет оформление проекта: вы создадите репозиторий на GitHub, оформите `README.md`, добавите артефакты (код, данные, отчеты) и напишете базовые тесты на `pytest`.


In [1]:
! pip install -q requests beautifulsoup4 schedule pytest


In [2]:
import json
import re
import time
from pathlib import Path
from urllib.parse import urljoin

import requests
import schedule
from bs4 import BeautifulSoup

CATALOGUE_URL = "http://books.toscrape.com/catalogue/page-{N}.html"
DATA_PATH = Path("artifacts") / "books_data.txt"
REQUEST_TIMEOUT_SECONDS = 30
_SESSION = requests.Session()


## Задание 1. Сбор данных об одной книге (2 баллов)

В этом задании мы начнем подготовку скрипта для парсинга информации о книгах со страниц каталога сайта [Books to Scrape](https://books.toscrape.com/).

Для начала реализуйте функцию `get_book_data`, которая будет получать данные о книге с одной страницы (например, с [этой](http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html)). Соберите всю информацию, включая название, цену, рейтинг, количество в наличии, описание и дополнительные характеристики из таблицы Product Information. Результат достаточно вернуть в виде словаря.

**Не забывайте про соблюдение PEP-8** — помимо качественно написанного кода важно также документировать функции по стандарту:
* кратко описать, что она делает и для чего нужна;
* какие входные аргументы принимает, какого они типа и что означают по смыслу;
* аналогично описать возвращаемые значения.

*P. S. Состав, количество аргументов функции и тип возвращаемого значения можете менять как вам удобно. То, что написано ниже в шаблоне — лишь пример.*


In [3]:
def _fetch_response(url: str) -> requests.Response:
    """Загружает страницу и кодировку.

    Args:
        url (str): абсолютный адрес страницы.

    Returns:
        requests.Response: ответ сервера.
        Текст читается в кодировке страницы.

    Raises:
        requests.RequestException: страница не загрузилась
        после трёх попыток.
    """
    last_error: requests.RequestException | None = None
    for attempt in range(3):
        try:
            response = _SESSION.get(
                url,
                timeout=REQUEST_TIMEOUT_SECONDS,
            )
            response.encoding = response.apparent_encoding or "utf-8"
            return response
        except requests.RequestException as error:
            last_error = error
            time.sleep(1 + attempt)
    if last_error is None:
        raise requests.RequestException(
            f"Не удалось загрузить {url}"
        )
    raise last_error


def _parse_stock_quantity(availability_text: str) -> int:
    """Достаёт число книг из строки наличия.

    Args:
        availability_text (str): текст вроде
        «In stock (22 available)».

    Returns:
        int: количество доступных экземпляров.
        0, если числа нет.
    """
    match = re.search(r"(\d+)", availability_text)
    if match is None:
        return 0
    return int(match.group(1))


def get_book_data(book_url: str) -> dict:
    """Собирает данные одной книги.

    Args:
        book_url (str): адрес страницы книги.

    Returns:
        dict: название, цена, рейтинг, количество
        в наличии, описание и все поля таблицы
        Product Information.
    """
    response = _fetch_response(book_url)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")

    title = soup.find("h1").get_text(strip=True)
    price = soup.select_one("p.price_color").get_text(strip=True)
    rating_class = soup.select_one("p.star-rating").get("class", [])
    rating = next(
        (name for name in rating_class if name != "star-rating"),
        "",
    )
    availability_text = soup.select_one(
        "p.availability"
    ).get_text(strip=True)
    description = ""
    description_header = soup.find(id="product_description")
    if description_header is not None:
        paragraph = description_header.find_next_sibling("p")
        if paragraph is not None:
            description = paragraph.get_text(strip=True)

    product_information: dict[str, str] = {}
    table = soup.find("table", class_="table")
    if table is not None:
        for row in table.find_all("tr"):
            header = row.find("th")
            value = row.find("td")
            if header is None or value is None:
                continue
            product_information[
                header.get_text(strip=True)
            ] = value.get_text(strip=True)

    return {
        "title": title,
        "price": price,
        "rating": rating,
        "stock_quantity": _parse_stock_quantity(availability_text),
        "description": description,
        "product_information": product_information,
    }


In [4]:
book_url = 'http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html'
get_book_data(book_url)


{'title': 'A Light in the Attic',
 'price': '£51.77',
 'rating': 'Three',
 'stock_quantity': 22,
 'description': "It's hard to imagine a world without A Light in the Attic. "
                'This now-classic collection of poetry and drawings from Shel '
                'Silverstein celebrates its 20th anniversary with this special '
                "edition. Silverstein's humorous and creative verse can amuse "
                'the dowdiest of readers. Lemon-faced adults and fidgety kids '
                'sit still and read these rhythmic words and laugh and smile '
                "and love th It's hard to imagine a world without A Light in "
                'the Attic. This now-classic collection of poetry and drawings '
                'from Shel Silverstein celebrates its 20th anniversary with '
                "this special edition. Silverstein's humorous and creative "
                'verse can amuse the dowdiest of readers. Lemon-faced adults '
                'and fidgety ki

## Задание 2. Сбор данных обо всех книгах (2 баллов)

Создайте функцию `scrape_books`, которая будет проходиться по всем страницам из каталога (вида `http://books.toscrape.com/catalogue/page-{N}.html`) и осуществлять парсинг всех страниц в цикле, используя ранее написанную `get_book_data`.

Добавьте аргумент-флаг, который будет отвечать за сохранение результата в файл: если он будет равен `True`, то информация сохранится в ту же папку в файл `books_data.txt`; иначе шаг сохранения будет пропущен.

**Также не забывайте про соблюдение PEP-8**


In [5]:
def _page_url(catalogue_url: str, page_number: int) -> str:
    """Подставляет номер страницы в шаблон.

    Args:
        catalogue_url (str): шаблон с маркером {N}.
        page_number (int): номер страницы, начиная с 1.

    Returns:
        str: адрес конкретной страницы каталога.
    """
    return catalogue_url.replace("{N}", str(page_number))


def _book_urls(page_url: str, page_html: str) -> list[str]:
    """Собирает абсолютные ссылки на книги.

    Args:
        page_url (str): адрес страницы каталога.
        База для относительных ссылок.
        page_html (str): HTML этой страницы.

    Returns:
        list[str]: ссылки на страницы книг
        в порядке показа на странице.
    """
    soup = BeautifulSoup(page_html, "html.parser")
    urls: list[str] = []
    for link in soup.select("article.product_pod h3 a"):
        href = link.get("href")
        if not href:
            continue
        urls.append(urljoin(page_url, href))
    return urls


def scrape_books(
    catalogue_url: str = CATALOGUE_URL,
    is_save: bool = False,
    max_pages: int | None = None,
) -> list[dict]:
    """Обходит каталог и собирает данные книг.

    Args:
        catalogue_url (str): шаблон адреса страницы.
        Номер страницы ставится вместо {N}.
        is_save (bool): True — записать результат
        в artifacts/books_data.txt.
        False — файл не создавать и не изменять.
        max_pages (int | None): сколько страниц обойти.
        None — все страницы, пока сайт их отдаёт.

    Returns:
        list[dict]: данные книг в порядке обхода.
        Каждый элемент — результат get_book_data.
    """
    books: list[dict] = []
    page_number = 1
    while True:
        if max_pages is not None and page_number > max_pages:
            break
        page_url = _page_url(catalogue_url, page_number)
        response = _fetch_response(page_url)
        if response.status_code == 404:
            break
        response.raise_for_status()
        book_urls = _book_urls(page_url, response.text)
        if not book_urls:
            break
        for book_url in book_urls:
            books.append(get_book_data(book_url))
        print(
            f"Страница {page_number}: "
            f"собрано {len(book_urls)} книг"
        )
        page_number += 1

    if is_save:
        DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
        DATA_PATH.write_text(
            json.dumps(books, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )
    return books


In [6]:
res = scrape_books(is_save=True)
print(type(res), len(res))


Страница 1: собрано 20 книг
Страница 2: собрано 20 книг
Страница 3: собрано 20 книг
Страница 4: собрано 20 книг
Страница 5: собрано 20 книг
Страница 6: собрано 20 книг
Страница 7: собрано 20 книг
Страница 8: собрано 20 книг
Страница 9: собрано 20 книг
Страница 10: собрано 20 книг
Страница 11: собрано 20 книг
Страница 12: собрано 20 книг
Страница 13: собрано 20 книг
Страница 14: собрано 20 книг
Страница 15: собрано 20 книг
Страница 16: собрано 20 книг
Страница 17: собрано 20 книг
Страница 18: собрано 20 книг
Страница 19: собрано 20 книг
Страница 20: собрано 20 книг
Страница 21: собрано 20 книг
Страница 22: собрано 20 книг
Страница 23: собрано 20 книг
Страница 24: собрано 20 книг
Страница 25: собрано 20 книг
Страница 26: собрано 20 книг
Страница 27: собрано 20 книг
Страница 28: собрано 20 книг
Страница 29: собрано 20 книг
Страница 30: собрано 20 книг
Страница 31: собрано 20 книг
Страница 32: собрано 20 книг
Страница 33: собрано 20 книг
Страница 34: собрано 20 книг
Страница 35: собрано 20

## Задание 3. Настройка регулярной выгрузки (1 баллов)

Настройте автоматический запуск функции сбора данных каждый день в 19:00.
Для автоматизации используйте библиотеку `schedule`. Функция должна запускаться в указанное время и сохранять обновленные данные в текстовый файл.



Бесконечный цикл должен обеспечивать постоянное ожидание времени для запуска задачи и выполнять ее по расписанию. Однако чтобы не перегружать систему, стоит подумать о том, чтобы выполнять проверку нужного времени не постоянно, а раз в какой-то промежуток. В этом вам может помочь `time.sleep(...)`.

Проверьте работоспособность кода локально на любом времени чч:мм.


In [7]:
from datetime import datetime, timedelta

schedule.clear()
daily = schedule.every().day.at("19:00").do(scrape_books, is_save=True)
print("Задача на 19:00:", daily)
print("Следующий запуск:", daily.next_run)

run_at = (datetime.now() + timedelta(seconds=3)).strftime("%H:%M:%S")
fired = {"done": False}


def short_job() -> None:
    """Отмечает, что короткая проверка расписания выполнена.

    Returns:
        None: печатает время задачи и ставит флаг выполнения.
    """
    fired["done"] = True
    print("Короткая проверка выполнена, время задачи:", run_at)


short = schedule.every().day.at(run_at).do(short_job)
print("Короткая проверка зарегистрирована на", run_at)
print(short)

deadline = time.time() + 10
while time.time() < deadline and not fired["done"]:
    schedule.run_pending()
    time.sleep(0.5)

schedule.clear()
print("Ячейка завершена. Короткая задача сработала:", fired["done"])


Задача на 19:00: Job(interval=1, unit=days, do=scrape_books, args=(), kwargs={'is_save': True})
Следующий запуск: 2026-11-13 19:00:00
Короткая проверка зарегистрирована на 12:44:09
Job(interval=1, unit=days, do=short_job, args=(), kwargs={})
Короткая проверка выполнена, время задачи: 12:44:09
Ячейка завершена. Короткая задача сработала: True


## Задание 4. Написание автотестов (1,5 баллов)

Создайте минимум три автотеста для ключевых функций парсинга — например, `get_book_data` и `scrape_books`. Идеи проверок (можете использовать свои):

* данные о книге возвращаются в виде словаря с нужными ключами;
* список ссылок или количество собранных книг соответствует ожиданиям;
* значения отдельных полей (например, `title`) корректны.

Оформите тесты в отдельном скрипте `tests/test_scraper.py`, используйте библиотеку `pytest`. Убедитесь, что тесты проходят успешно при запуске из терминала командой `pytest`.

Также выведите результат их выполнения в ячейке ниже.

**Не забывайте про соблюдение PEP-8**


In [8]:
! pytest tests/test_scraper.py


============================= test session starts ==============================
platform darwin -- Python 3.14.6, pytest-9.2.0, pluggy-1.7.0
rootdir: /Users/dmitriygrosh/Documents/books-scraper
collected 3 items

tests/test_scraper.py ...                                                [100%]

============================== 3 passed in 4.46s ===============================


## Задание 5. Оформление проекта на GitHub и работа с Git (3,5 баллов)

В этом задании нужно воспользоваться системой контроля версий Git и платформой GitHub для хранения и управления своим проектом. **Ссылку на свой репозиторий пришлите в форме для сдачи ответа.**

### Пошаговая инструкция и задания

**1. Установите Git на свой компьютер.**

* Для Windows: [скачайте установщик](https://git-scm.com/downloads) и выполните установку.
* Для macOS:

  ```
  brew install git
  ```
* Для Linux:

  ```
  sudo apt update
  sudo apt install git
  ```

**2. Настройте имя пользователя и email.**

Это нужно для подписи ваших коммитов, сделайте в терминале через `git config ...`.

**3. Создайте аккаунт на GitHub**, если у вас его еще нет:
[https://github.com](https://github.com)

**4. Создайте новый репозиторий на GitHub:**

* Найдите кнопку **New repository**.
* Укажите название, краткое описание, выберите тип **Public** (чтобы мы могли проверить ДЗ).
* Не ставьте галочку Initialize this repository with a README.

**5. Создайте локальную папку с проектом.** Можно в терминале, можно через UI, это не имеет значения.

**6. Инициализируйте Git в этой папке.** Здесь уже придется воспользоваться некоторой командой в терминале.

**7. Привяжите локальный репозиторий к удаленному на GitHub.**

**8. Создайте ветку разработки.** По умолчанию вы будете находиться в ветке `main`, создайте и переключитесь на ветку `hw-books-parser`.

**9. Добавьте в проект следующие файлы и папки:**

* `scraper.py` — ваш основной скрипт для сбора данных.
* `README.md` — файл с кратким описанием проекта:

  * цель;
  * инструкции по запуску;
  * список используемых библиотек.
* `requirements.txt` — файл со списком зависимостей, необходимых для проекта (не присылайте все из глобального окружения, создайте изолированную виртуальную среду, добавьте в нее все нужное для проекта и получите список библиотек через `pip freeze`).
* `artifacts/` — папка с результатами парсинга (`books_data.txt` — полностью или его часть, если весь не поместится на GitHub).
* `notebooks/` — папка с заполненным ноутбуком `HW_03_python_ds.ipynb` и запущенными ячейками с выводами на экран.
* `tests/` — папка с тестами на `pytest`, оформите их в формате скрипта(-ов) с расширением `.py`.
* `.gitignore` — стандартный файл, который позволит исключить временные файлы при добавлении в отслеживаемые (например, `__pycache__/`, `.DS_Store`, `*.pyc`, `venv/` и др.).


**10. Сделайте коммит.**

**11. Отправьте свою ветку на GitHub.**

**12. Создайте Pull Request:**

* Перейдите в репозиторий на GitHub.
* Нажмите кнопку **Compare & pull request**.
* Укажите, что было добавлено, и нажмите **Create pull request**.

**13. Выполните слияние Pull Request:**

* Убедитесь, что нет конфликтов.
* Нажмите **Merge pull request**, затем **Confirm merge**.

**14. Скачайте изменения из основной ветки локально.**



### Требования к итоговому репозиторию

* Файл `scraper.py` с рабочим кодом парсера.
* `README.md` с описанием проекта и инструкцией по запуску.
* Папка `artifacts/` с результатом сбора данных (`.txt` файл).
* Папка `tests/` с тестами на `pytest`.
* Папка `notebooks/` с заполненным ноутбуком `HW_03_python_ds_2025.ipynb`.
* Pull Request с комментарием из ветки `hw-books-parser` в ветку `main`.
* Примерная структура:

  ```
  books_scraper/
  ├── artifacts/
  │   └── books_data.txt
  ├── notebooks/
  │   └── HW_03_python_ds_2025.ipynb
  ├── scraper.py
  ├── README.md
  ├── tests/
  │   └── test_scraper.py
  ├── .gitignore
  └── requirements.txt
  ```


## Выводы, рефлексия и сдача

### Рефлексия после выполнения (заполните здесь)

**1. Что было задано и что получилось**:

Ответ: нужно было снять каталог books.toscrape.com: функция одной книги, обход всех страниц с сохранением в файл, ежедневный запуск в 19:00, минимум три теста pytest и оформленный репозиторий. Скрипт это делает. `get_book_data` возвращает название, цену, рейтинг, число в наличии, описание и всю таблицу Product Information. `scrape_books(is_save=True)` собрал 1000 книг, в выводе ячейки `<class 'list'> 1000`, файл лежит в `artifacts/books_data.txt`. Тесты `pytest tests/test_scraper.py` прошли: 3 passed. Бесконечный цикл с `time.sleep` стоит в `scraper.py` под `if __name__`, в ноутбуке ячейка только регистрирует 19:00 и проверяет ближайшие секунды, поэтому она заканчивается.

**2. Как выполняли**: что сработало сразу, где застряли и как выбрались:

Ответ: разметка страницы читается прямо в HTML: заголовок в `h1`, цена в `p.price_color`, рейтинг в классе `star-rating`, описание — соседний абзац после `#product_description`, характеристики — строки `table`. Это собралось с первого прохода по примеру A Light in the Attic. Застрял на кодировке: у сайта в `Content-Type` нет charset, `requests` читает страницу как ISO-8859-1, и фунт в цене портится. Помогло `response.encoding = response.apparent_encoding`. Второй затык — pytest 9. Команда `pytest tests/test_scraper.py` не видела `scraper.py`, потому что в `sys.path` попадал каталог `tests`, а не корень проекта. В начале `tests/test_scraper.py` добавил корень в `sys.path`. В шаблоне ячейки опечатка `pytest test/test_scraper.py`, запускаю `tests/test_scraper.py`.

**3. Использование ИИ**: цель, использованные промпты, где помощник ошибся и как вы это поняли. Если не использовали, так и напишите:

Ответ: использовал, чтобы проверить, всё ли по заданию уже сделано и ничего ли не забыто. Просил сверить решение с материалами курса до модуля 9. По сверке ноутбук не совпадал с шаблоном `HW_03_python_ds.ipynb`: формулировки заданий были переписаны, а вопросы рефлексии были другие. Ноутбук собрал заново по шаблону, ячейки оставил с выводом. Отдельно помощник сам сделал коммит, push и merge pull request, хотя git я проводил сам. Лишнюю ветку убрал. Смерженный PR GitHub не удаляет, это видно по ответу API.

**4. Что осталось непонятным** (честный список, он поможет на консультации):

Ответ: что именно значит в задании 2 «сохранить в ту же папку в файл books_data.txt», если в задании 5 файл уже лежит в `artifacts/`. Я сохранил в `artifacts/books_data.txt`, как в итоговой структуре. Непонятно, какое имя ноутбука считать верным: в форме сдачи `ФИО_HW_03_python_ds`, внизу шаблона ещё и `HW_03_python_ds_2025.ipynb`. У меня файл `GROSH_D_A_HW_03_python_ds.ipynb`. И не до конца понимаю, сработает ли `schedule` на 19:00, если в этот момент скрипт не запущен: библиотека ничего не догоняет задним числом.

**5. Самооценка по критериям задания**:

Ответ: задание 1 — 2 из 2, карточка книги собирается. Задание 2 — 2 из 2, обойдены все 50 страниц, 1000 книг записаны. Задание 3 — 1 из 1, задача на 19:00 зарегистрирована, пауза через `time.sleep` есть, короткая проверка в ноутбуке дошла до конца. Задание 4 — 1,5 из 1,5, три теста проходят. Задание 5 ставлю частично: README, requirements из своего venv, `.gitignore`, artifacts и тесты на месте, но отдельной ветки `hw-books-parser` и нового pull request в `main` я ещё не оформил. Коммит в `main` уже есть, это не тот порядок, который расписан в шагах 8–14.

**6. Дополнительный вопрос: обратная связь по заданию.** Что было ценно, что сложно, а что, на ваш взгляд, требует доработки?

Ответ: ценно, что парсинг, pytest и git собраны в одном небольшом проекте на учебном сайте, а не на трёх разных. Сложно было не написать цикл по страницам, а понять, какой из двух текстов в шаблоне главный: путь к файлу, имя ноутбука и команда pytest в разных местах расходятся. Доработал бы шаблон: один путь `artifacts/books_data.txt`, одно имя ноутбука и сразу правильная команда `pytest tests/test_scraper.py`.

> Напоминание: рефлексия пишется своими словами, без помощи ИИ.
